# Neural Networks — Implementations

The two-layer MLP twice more: once on tensors with `torch.autograd` replacing the hand-derived backprop, once as the standard `nn.Sequential` training loop with the weights copied from the very same NumPy Xavier draw. All randomness flows through `np.random.default_rng`, everything stays float64, and the step count is fixed, so the three lanes walk the same trajectory to ~1e-12.

## 13_mlp_classifier

One hidden layer of tanh units turns XOR from impossible to easy.

### torch

The identical network and update rule, with the backward pass handed to autograd: the loss is written as `log_softmax` (the same objective — the notebook's `+1e-12` inside the log is a numerical guard that its own hand-derived gradient ignores), so `autograd.grad` reproduces `(P − Y)/n` and the chain rule exactly. **What torch adds:** the entire backward pass for free, plus a machine check of the notebook's hand gradients.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Draw W1 then W2 with np.random.default_rng(random_state), same order as NumPy.
# 2. Write the loss with log_softmax: its gradient w.r.t. scores is exactly (P - Y)/n.
# 3. autograd.grad(loss, [W1, b1, W2, b2]) returns the whole backward pass at once.
# 4. Update the parameters inside torch.no_grad(); autograd must not trace the step.
# 5. float64 everywhere - float32 turns 1e-13 cross-lane agreement into 1e-5.


class MLPClassifierScratch:
    """Two-layer tanh MLP on tensors; autograd replaces the hand-written backprop.

    Same interface and the same Xavier draws as the NumPy lane: the weights come
    from np.random.default_rng(random_state), so the trajectories match exactly.
    """

    def __init__(self, n_hidden=8, learning_rate=0.15, n_steps=2000,
                 weight_decay=0.0, random_state=42):
        self.n_hidden = n_hidden
        self.learning_rate = learning_rate
        self.n_steps = n_steps
        self.weight_decay = weight_decay
        self.random_state = random_state

    def _initialize_parameters(self, d, c):
        rng_init = np.random.default_rng(self.random_state)
        h = self.n_hidden
        std1 = np.sqrt(2.0 / (d + h))
        std2 = np.sqrt(2.0 / (h + c))
        self.W1_ = torch.as_tensor(rng_init.normal(0, std1, size=(d, h))).requires_grad_(True)
        self.b1_ = torch.zeros(h, dtype=torch.float64, requires_grad=True)
        self.W2_ = torch.as_tensor(rng_init.normal(0, std2, size=(h, c))).requires_grad_(True)
        self.b2_ = torch.zeros(c, dtype=torch.float64, requires_grad=True)

    def _forward(self, X):
        Z1 = X @ self.W1_ + self.b1_
        H = torch.tanh(Z1)
        S = H @ self.W2_ + self.b2_
        P = torch.softmax(S, dim=1)
        cache = (X, Z1, H, S, P)
        return P, cache

    def _loss_and_grads(self, cache, Y):
        X, Z1, H, S, P = cache
        # log_softmax is the notebook's objective; its gradient w.r.t. S is
        # exactly (P - Y)/n - the very line the hand-derived backprop starts from.
        # (The notebook's +1e-12 inside log is a guard its own gradient ignores.)
        loss_data = -(Y * torch.log_softmax(S, dim=1)).sum(dim=1).mean()
        loss_reg = 0.5 * self.weight_decay * ((self.W1_ ** 2).sum() + (self.W2_ ** 2).sum())
        loss = loss_data + loss_reg
        gW1, gb1, gW2, gb2 = torch.autograd.grad(
            loss, [self.W1_, self.b1_, self.W2_, self.b2_])
        grads = {'W1': gW1, 'b1': gb1, 'W2': gW2, 'b2': gb2}
        return float(loss.detach()), grads

    def _check_is_fitted(self):
        if not hasattr(self, 'W1_'):
            raise RuntimeError('Call fit() before predict.')

    def fit(self, X, y):
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        y = np.asarray(y, dtype=int)
        self.classes_ = np.unique(y)
        c = len(self.classes_)
        yt = torch.as_tensor(y)
        Y = torch.zeros((len(y), c), dtype=torch.float64)
        Y[torch.arange(len(y)), yt] = 1.0
        self._initialize_parameters(Xt.shape[1], c)

        self.history_ = {'loss': [], 'accuracy': []}
        for step in range(self.n_steps):
            P, cache = self._forward(Xt)
            loss, grads = self._loss_and_grads(cache, Y)
            self.history_['loss'].append(loss)
            self.history_['accuracy'].append(float((P.argmax(dim=1) == yt).double().mean()))

            with torch.no_grad():
                self.W1_ -= self.learning_rate * grads['W1']
                self.b1_ -= self.learning_rate * grads['b1']
                self.W2_ -= self.learning_rate * grads['W2']
                self.b2_ -= self.learning_rate * grads['b2']

        return self

    def predict_proba(self, X):
        self._check_is_fitted()
        with torch.no_grad():
            P, _ = self._forward(torch.as_tensor(np.asarray(X, dtype=float)))
        return P.numpy()

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

    def score(self, X, y):
        return float(np.mean(self.predict(X) == np.asarray(y)))


In [ ]:
# exports: final_loss, proba_head, preds, train_acc
_rng_eq = np.random.default_rng(77)
X_eq = _rng_eq.uniform(-1.5, 1.5, size=(60, 2))
y_eq = ((X_eq[:, 0] * X_eq[:, 1]) > 0).astype(int)
X_eq = X_eq + _rng_eq.normal(0, 0.2, size=X_eq.shape)

_mlp_eq = MLPClassifierScratch(n_hidden=6, learning_rate=0.2, n_steps=400,
                               weight_decay=0.001, random_state=3).fit(X_eq, y_eq)
final_loss = _mlp_eq.history_['loss'][-1]
proba_head = _mlp_eq.predict_proba(X_eq[:8])
preds = _mlp_eq.predict(X_eq)
train_acc = _mlp_eq.score(X_eq, y_eq)
print(f"final loss {final_loss:.6f}   train accuracy {train_acc:.1%}")


In [ ]:
# Autograd reproduces the hand-derived output-layer gradient: db2 = sum((P-Y)/n).
_P_c, _cache_c = _mlp_eq._forward(torch.as_tensor(X_eq))
_Y_c = torch.zeros((len(y_eq), 2), dtype=torch.float64)
_Y_c[torch.arange(len(y_eq)), torch.as_tensor(y_eq)] = 1.0
_, _grads_c = _mlp_eq._loss_and_grads(_cache_c, _Y_c)
_db2_hand = ((_P_c.detach() - _Y_c) / len(y_eq)).sum(dim=0)
assert torch.allclose(_grads_c['b2'], _db2_hand, atol=1e-12), \
    "autograd must agree with the notebook's (P - Y)/n backprop"
assert _mlp_eq.history_['loss'][-1] < _mlp_eq.history_['loss'][0], "training must reduce the loss"
assert np.max(np.abs(proba_head.sum(axis=1) - 1.0)) < 1e-12, "softmax rows sum to one"

# The hidden layer earns its keep: a linear classifier is near chance on XOR.
_Xa_c = np.column_stack([np.ones(len(X_eq)), X_eq])
_wlin_c = np.linalg.lstsq(_Xa_c, y_eq, rcond=None)[0]
_acc_lin = float(np.mean(((_Xa_c @ _wlin_c) > 0.5).astype(int) == y_eq))
assert train_acc > _acc_lin + 0.2, "the MLP must clearly beat a linear classifier"


### library

`nn.Sequential(Linear, Tanh, Linear)` trained with `nn.CrossEntropyLoss` and `torch.optim.SGD`, initialised from the identical NumPy Xavier draw — `nn.Linear` keeps its weight as `(out, in)`, so the copy is transposed. `SGD(weight_decay=λ)` adds `λ·W` to the gradient, which is exactly the notebook's `0.5·λ‖W‖²` penalty, applied to weights only. **What the library adds:** the standard model/criterion/optimiser vocabulary for the same arithmetic.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# hints:
# 1. nn.Linear stores weight as (out, in): copy the NumPy (in, out) draw transposed.
# 2. CrossEntropyLoss eats raw logits and integer labels - no softmax layer in the model.
# 3. SGD(weight_decay=wd) adds wd*W to the grad: exactly the 0.5*wd*||W||^2 penalty here.
# 4. Biases get their own param group with weight_decay=0; the notebook never decays b.
# 5. Build the Linears with dtype=torch.float64 before copying the float64 draws in.


class MLPClassifierScratch:
    """The same MLP as nn.Sequential trained by CrossEntropyLoss + optim.SGD.

    The weights start from the identical NumPy Xavier draw (transposed into
    nn.Linear's (out, in) layout), so every update matches the scratch lanes.
    """

    def __init__(self, n_hidden=8, learning_rate=0.15, n_steps=2000,
                 weight_decay=0.0, random_state=42):
        self.n_hidden = n_hidden
        self.learning_rate = learning_rate
        self.n_steps = n_steps
        self.weight_decay = weight_decay
        self.random_state = random_state

    def _initialize_parameters(self, d, c):
        rng_init = np.random.default_rng(self.random_state)
        h = self.n_hidden
        std1 = np.sqrt(2.0 / (d + h))
        std2 = np.sqrt(2.0 / (h + c))
        self.model_ = nn.Sequential(
            nn.Linear(d, h, dtype=torch.float64),
            nn.Tanh(),
            nn.Linear(h, c, dtype=torch.float64),
        )
        with torch.no_grad():
            self.model_[0].weight.copy_(torch.as_tensor(rng_init.normal(0, std1, size=(d, h)).T))
            self.model_[0].bias.zero_()
            self.model_[2].weight.copy_(torch.as_tensor(rng_init.normal(0, std2, size=(h, c)).T))
            self.model_[2].bias.zero_()

    def _forward(self, X):
        S = self.model_(X)
        P = torch.softmax(S, dim=1)
        return P, S

    def _check_is_fitted(self):
        if not hasattr(self, 'model_'):
            raise RuntimeError('Call fit() before predict.')

    def fit(self, X, y):
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        y = np.asarray(y, dtype=int)
        self.classes_ = np.unique(y)
        yt = torch.as_tensor(y)
        self._initialize_parameters(Xt.shape[1], len(self.classes_))

        weights = [self.model_[0].weight, self.model_[2].weight]
        biases = [self.model_[0].bias, self.model_[2].bias]
        optimizer = torch.optim.SGD(
            [{'params': weights, 'weight_decay': self.weight_decay},
             {'params': biases, 'weight_decay': 0.0}],
            lr=self.learning_rate)
        criterion = nn.CrossEntropyLoss()

        self.history_ = {'loss': [], 'accuracy': []}
        for step in range(self.n_steps):
            logits = self.model_(Xt)
            loss_data = criterion(logits, yt)
            with torch.no_grad():
                reg = 0.5 * self.weight_decay * sum(float((w ** 2).sum()) for w in weights)
                self.history_['loss'].append(float(loss_data.detach()) + reg)
                self.history_['accuracy'].append(float((logits.argmax(dim=1) == yt).double().mean()))
            optimizer.zero_grad()
            loss_data.backward()
            optimizer.step()

        return self

    def predict_proba(self, X):
        self._check_is_fitted()
        with torch.no_grad():
            P, _ = self._forward(torch.as_tensor(np.asarray(X, dtype=float)))
        return P.numpy()

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

    def score(self, X, y):
        return float(np.mean(self.predict(X) == np.asarray(y)))


In [ ]:
# exports: final_loss, proba_head, preds, train_acc
_rng_eq = np.random.default_rng(77)
X_eq = _rng_eq.uniform(-1.5, 1.5, size=(60, 2))
y_eq = ((X_eq[:, 0] * X_eq[:, 1]) > 0).astype(int)
X_eq = X_eq + _rng_eq.normal(0, 0.2, size=X_eq.shape)

_mlp_eq = MLPClassifierScratch(n_hidden=6, learning_rate=0.2, n_steps=400,
                               weight_decay=0.001, random_state=3).fit(X_eq, y_eq)
final_loss = _mlp_eq.history_['loss'][-1]
proba_head = _mlp_eq.predict_proba(X_eq[:8])
preds = _mlp_eq.predict(X_eq)
train_acc = _mlp_eq.score(X_eq, y_eq)
print(f"final loss {final_loss:.6f}   train accuracy {train_acc:.1%}")


In [ ]:
# The init really is the same draw: nn.Linear stores (out, in), so W1.T lands there.
_probe = MLPClassifierScratch(n_hidden=6, random_state=3)
_probe._initialize_parameters(2, 2)
_rng_c = np.random.default_rng(3)
_W1_draw = _rng_c.normal(0, np.sqrt(2.0 / (2 + 6)), size=(2, 6))
assert np.allclose(_probe.model_[0].weight.detach().numpy(), _W1_draw.T, rtol=0, atol=1e-15), \
    "layer-0 weight must be the NumPy Xavier draw, transposed"

# weight_decay in optim.SGD is the notebook's L2 term: more decay, smaller weights.
_heavy = MLPClassifierScratch(n_hidden=6, learning_rate=0.2, n_steps=400,
                              weight_decay=0.05, random_state=3).fit(X_eq, y_eq)
_wn = float(sum((w.detach() ** 2).sum() for w in [_mlp_eq.model_[0].weight, _mlp_eq.model_[2].weight]))
_wn_heavy = float(sum((w.detach() ** 2).sum() for w in [_heavy.model_[0].weight, _heavy.model_[2].weight]))
assert _wn_heavy < _wn, "raising weight_decay must shrink the trained weights"
assert _mlp_eq.history_['loss'][-1] < _mlp_eq.history_['loss'][0], "training must reduce the loss"
assert train_acc > 0.8, "the wrapped nn.Sequential still solves noisy XOR"
